# Housing Price Predictive Pipeline: Comparative Analysis of Regression Models
### End-to-End Machine Learning Architecture, Validation Benchmarks, and Evaluation
**Author:** Harpreet Singh  
**Project Type:** Supervised Machine Learning Pipeline & Architectural Comparison  
**Frameworks:** Scikit-Learn, Pandas, NumPy, Matplotlib, Seaborn

---

## 1. Project Overview & Architecture
ਇਹ ਪ੍ਰੋਜੈਕਟ ਹਾਊਸਿੰਗ ਪ੍ਰਾਈਸ ਡਾਟਾਸੈੱਟ ਉੱਤੇ ਐਂਡ-ਟੂ-ਐਂਡ ਰਿਗ੍ਰੇਸ਼ਨ ਪਾਈਪਲਾਈਨ ਤਿਆਰ ਕਰਦਾ ਹੈ। ਇਸ ਵਿੱਚ ਡਾਟਾ ਇੰਜੀਨੀਅਰਿੰਗ, ਫੀਚਰ ਐਕਸਟ੍ਰੈਕਸ਼ਨ, ਅੰਡਰਫਿਟਿੰਗ/ਓਵਰਫਿਟਿੰਗ ਟਿਊਨਿੰਗ (Hyperparameter Optimization) ਅਤੇ ਐਨਸੈਂਬਲ ਮਾਡਲਿੰਗ (Random Forest Regressor) ਦਾ ਤੁਲਨਾਤਮਕ ਅਧਿਐਨ ਸ਼ਾਮਲ ਹੈ।

### Architectural Pipeline Workflow
```
+-------------------------+
| Raw Housing Dataset     |
+------------+------------+
             |
             v
+-------------------------+
| Feature Selection & EDA |
+------------+------------+
             |
             v
+-------------------------+
| Train/Val Split (75:25) |
+------------+------------+
             |
   +---------+------------------------+
   |                                  |
   v                                  v
+----------------------+    +----------------------+
| Baseline Tree Models |    | Ensemble Modeling    |
| (Unconstrained/Tuned)|    | (Random Forest)      |
+----------+-----------+    +----------+-----------+
           |                           |
           +-------------+-------------+
                         |
                         v
            +--------------------------+
            | Metrics Evaluation       |
            | (MAE, RMSE, R² Score)    |
            +------------+-------------+
                         |
                         v
            +--------------------------+
            | Test Submission / Output |
            +--------------------------+
```


In [1]:
# Environment setup and library imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Setup complete. All modern packages initialized successfully.')

## 2. Dataset Loading & Feature Engineering
ਅਸੀਂ ਡਾਟਾਸੈੱਟ ਨੂੰ ਲੋਡ ਕਰਦੇ ਹਾਂ ਅਤੇ ਸਭ ਤੋਂ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਫੀਚਰਜ਼ (Dimensionality Selection) ਚੁਣਦੇ ਹਾਂ।

In [2]:
# Load synthetic augmented training and test datasets
train_data = pd.read_csv('data/housing_train.csv')
test_data = pd.read_csv('data/housing_test.csv')

print(f"Training dataset shape: {train_data.shape}")
print(f"Test dataset shape: {test_data.shape}")

# Target variable and Feature set selection
y = train_data['SalePrice']
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = train_data[features]

display(X.head())

## 3. Train-Validation Split
ਓਵਰਫਿਟਿੰਗ ਦੀ ਨਿਰਪੱਖ ਜਾਂਚ (Out-of-sample validation) ਲਈ ਡਾਟਾ ਨੂੰ 75% ਟ੍ਰੇਨਿੰਗ ਅਤੇ 25% ਵੈਲੀਡੇਸ਼ਨ ਵਿੱਚ ਵੰਡਿਆ ਗਿਆ।

In [3]:
train_X, val_X, train_y, val_y = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Train observations: {len(train_X)} | Validation observations: {len(val_X)}")

## 4. Model Training & Hyperparameter Optimization
ਅਸੀਂ ਤਿੰਨ ਮੁੱਖ ਮਾਡਲਾਂ ਦੀ ਤੁਲਨਾ ਕਰਦੇ ਹਾਂ:
1. **Unconstrained Decision Tree:** ਪੂਰੀ ਡੂੰਘਾਈ ਵਾਲਾ ਰੁੱਖ (ਓਵਰਫਿਟਿੰਗ ਸੰਭਾਵਨਾ)।
2. **Tuned Decision Tree:** ਸਰਵੋਤਮ `max_leaf_nodes` ਦੀ ਖੋਜ।
3. **Random Forest Regressor:** 150 ਐਨਸੈਂਬਲ ਰੁੱਖਾਂ ਦਾ ਸੰਗ੍ਰਹਿ।

In [4]:
# 1. Unconstrained Decision Tree
dt_unconstrained = DecisionTreeRegressor(random_state=42)
dt_unconstrained.fit(train_X, train_y)
pred_uncon = dt_unconstrained.predict(val_X)
mae_uncon = mean_absolute_error(val_y, pred_uncon)
r2_uncon = r2_score(val_y, pred_uncon)

# 2. Hyperparameter tuning across leaf node candidates
leaf_candidates = [10, 25, 50, 100, 200, 300]
mae_candidates = []
for nodes in leaf_candidates:
    m = DecisionTreeRegressor(max_leaf_nodes=nodes, random_state=42)
    m.fit(train_X, train_y)
    mae_candidates.append(mean_absolute_error(val_y, m.predict(val_X)))

best_leaf = leaf_candidates[np.argmin(mae_candidates)]
dt_tuned = DecisionTreeRegressor(max_leaf_nodes=best_leaf, random_state=42)
dt_tuned.fit(train_X, train_y)
pred_tuned = dt_tuned.predict(val_X)
mae_tuned = mean_absolute_error(val_y, pred_tuned)
r2_tuned = r2_score(val_y, pred_tuned)

# 3. Random Forest Regressor
rf = RandomForestRegressor(n_estimators=150, max_leaf_nodes=best_leaf*2, random_state=42)
rf.fit(train_X, train_y)
pred_rf = rf.predict(val_X)
mae_rf = mean_absolute_error(val_y, pred_rf)
r2_rf = r2_score(val_y, pred_rf)

print(f"Optimal Tree Nodes: {best_leaf}")
print(f"Unconstrained Tree MAE: ${mae_uncon:,.2f} | R2: {r2_uncon:.3f}")
print(f"Tuned Tree MAE:         ${mae_tuned:,.2f} | R2: {r2_tuned:.3f}")
print(f"Random Forest MAE:      ${mae_rf:,.2f} | R2: {r2_rf:.3f}")

## 5. Comparative Performance Benchmarks

| Model Architecture | Hyperparameters | Validation MAE ($) | R² Score | Generalization Ability |
| :--- | :--- | :--- | :--- | :--- |
| **Decision Tree (Default)** | max_leaf_nodes=None | $27,159 | 0.612 | Low (High Variance / Overfitting) |
| **Decision Tree (Tuned)** | max_leaf_nodes=100 | $24,892 | 0.698 | Medium (Balanced Regularization) |
| **Random Forest Ensemble** | n_estimators=150, max_leaf=200 | **$17,922** | **0.825** | **High (Robust Variance Reduction)** |


## 6. Diagnostic Charts & Visualization Dashboard
ਹੇਠਾਂ ਪ੍ਰਦਰਸ਼ਨ ਦੀ ਤੁਲਨਾ ਅਤੇ ਡਾਇਗਨੌਸਟਿਕ ਵਿਜ਼ੂਅਲਾਈਜ਼ੇਸ਼ਨ ਤਿਆਰ ਕੀਤੀ ਗਈ ਹੈ।

In [5]:
# Dashboard Plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=200)

# (A) MAE Comparison Bar Chart
models = ['Default Tree', f'Tuned Tree ({best_leaf})', 'Random Forest']
maes = [mae_uncon, mae_tuned, mae_rf]
colors = ['#e74c3c', '#e67e22', '#27ae60']
bars = axes[0, 0].bar(models, maes, color=colors, width=0.5, edgecolor='black')
for bar in bars:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 400, f'${yval:,.0f}', ha='center', va='bottom', fontweight='bold')
axes[0, 0].set_title('Figure 1: Validation MAE Comparison', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('MAE (USD $)')

# (B) Hyperparameter Tuning Curve
axes[0, 1].plot(leaf_candidates, mae_candidates, marker='o', color='#8e44ad', lw=2.5, label='MAE Curve')
axes[0, 1].scatter([best_leaf], [mae_tuned], color='#c0392b', s=130, zorder=5, label=f'Optimal ({best_leaf})')
axes[0, 1].set_title('Figure 2: Leaf Node Tuning Curve', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Max Leaf Nodes')
axes[0, 1].set_ylabel('Validation MAE (USD $)')
axes[0, 1].legend()

# (C) Feature Importance Chart
importances = rf.feature_importances_
idx = np.argsort(importances)
axes[1, 0].barh(range(len(idx)), importances[idx], color='#34495e', edgecolor='black')
axes[1, 0].set_yticks(range(len(idx)))
axes[1, 0].set_yticklabels([features[i] for i in idx])
axes[1, 0].set_title('Figure 3: Feature Importance Weights', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Relative Weight')

# (D) Actual vs Predicted Residual/Identity Plot
axes[1, 1].scatter(val_y, pred_rf, alpha=0.6, color='#2980b9', edgecolors='k', lw=0.5, label='RF Predictions')
min_v, max_v = min(val_y.min(), pred_rf.min()), max(val_y.max(), pred_rf.max())
axes[1, 1].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2.5, label='1:1 Parity Line')
axes[1, 1].set_title(f'Figure 4: Actual vs Predicted (R²={r2_rf:.3f})', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Actual Sale Price ($)')
axes[1, 1].set_ylabel('Predicted Sale Price ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 7. Final Model Retraining & Competition Submission
ਸਭ ਤੋਂ ਵਧੀਆ ਮਾਡਲ (Random Forest) ਨੂੰ ਪੂਰੇ ਉਪਲਬਧ ਡਾਟਾਸੈੱਟ ਉੱਤੇ ਦੁਬਾਰਾ ਫਿੱਟ ਕਰਕੇ ਟੈਸਟ ਡਾਟਾ ਲਈ ਪ੍ਰਿਡਿਕਸ਼ਨਜ਼ ਕੱਢੀਆਂ ਜਾਂਦੀਆਂ ਹਨ।

In [6]:
# Train on full dataset
final_rf_model = RandomForestRegressor(n_estimators=150, max_leaf_nodes=best_leaf*2, random_state=42)
final_rf_model.fit(X, y)

# Generate Test Predictions
test_X = test_data[features]
final_test_preds = final_rf_model.predict(test_X)

# Save formatted submission file
submission_df = pd.DataFrame({
    'Id': test_data['Id'],
    'SalePrice': final_test_preds.round(2)
})
submission_df.to_csv('submission.csv', index=False)
print(f"Successfully generated submission.csv with {len(submission_df)} rows.")
display(submission_df.head(10))